# 05 ML GNN Embeddings — Group 1: Link-Prediction Embeddings (Log Target)

**Group 1.** Trains regressors on GNN embeddings produced by the link-prediction training objective — no reconstruction loss, no fine-tuning. Target is `log_systemic_risk_label`.

| Dataset | Model | Dim |
|---|---|---|
| `graphsage_v1_32_srisk_dataset.parquet` | GraphSAGE v1 | 32 |
| `graphsage_v1_64_srisk_dataset.parquet` | GraphSAGE v1 | 64 |
| `graphsage_v1_128_srisk_dataset.parquet` | GraphSAGE v1 | 128 |
| `node2vec_v1_32_srisk_dataset.parquet` | Node2Vec v1 | 32 |
| `node2vec_v1_64_srisk_dataset.parquet` | Node2Vec v1 | 64 |
| `node2vec_v1_128_srisk_dataset.parquet` | Node2Vec v1 | 128 |

> Run `03_g1_ref.ipynb` first to generate the parquet files.

In [1]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = find_project_root()

In [2]:
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Datasets

In [3]:
df_sage_32, feature_cols_sage_32 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="graphsage_v1_32_srisk_dataset.parquet")
print(f"GraphSAGE v1 32:  {df_sage_32.shape}  —  {len(feature_cols_sage_32)} embedding cols")

df_sage_64, feature_cols_sage_64 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="graphsage_v1_64_srisk_dataset.parquet")
print(f"GraphSAGE v1 64:  {df_sage_64.shape}  —  {len(feature_cols_sage_64)} embedding cols")

df_sage_128, feature_cols_sage_128 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="graphsage_v1_128_srisk_dataset.parquet")
print(f"GraphSAGE v1 128: {df_sage_128.shape}  —  {len(feature_cols_sage_128)} embedding cols")

df_n2v_32, feature_cols_n2v_32 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_v1_32_srisk_dataset.parquet")
print(f"Node2Vec v1 32:   {df_n2v_32.shape}  —  {len(feature_cols_n2v_32)} embedding cols")

df_n2v_64, feature_cols_n2v_64 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_v1_64_srisk_dataset.parquet")
print(f"Node2Vec v1 64:   {df_n2v_64.shape}  —  {len(feature_cols_n2v_64)} embedding cols")

df_n2v_128, feature_cols_n2v_128 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_v1_128_srisk_dataset.parquet")
print(f"Node2Vec v1 128:  {df_n2v_128.shape}  —  {len(feature_cols_n2v_128)} embedding cols")

GraphSAGE v1 32:  (145536, 37)  —  32 embedding cols
GraphSAGE v1 64:  (145536, 69)  —  64 embedding cols
GraphSAGE v1 128: (145536, 133)  —  128 embedding cols
Node2Vec v1 32:   (145536, 37)  —  32 embedding cols
Node2Vec v1 64:   (145536, 69)  —  64 embedding cols
Node2Vec v1 128:  (145536, 133)  —  128 embedding cols


In [4]:
trainer_sage_32  = ModelTrainer(df=df_sage_32,  feature_cols=feature_cols_sage_32,  target_col="log_systemic_risk_label")
trainer_sage_64  = ModelTrainer(df=df_sage_64,  feature_cols=feature_cols_sage_64,  target_col="log_systemic_risk_label")
trainer_sage_128 = ModelTrainer(df=df_sage_128, feature_cols=feature_cols_sage_128, target_col="log_systemic_risk_label")
trainer_n2v_32   = ModelTrainer(df=df_n2v_32,   feature_cols=feature_cols_n2v_32,   target_col="log_systemic_risk_label")
trainer_n2v_64   = ModelTrainer(df=df_n2v_64,   feature_cols=feature_cols_n2v_64,   target_col="log_systemic_risk_label")
trainer_n2v_128  = ModelTrainer(df=df_n2v_128,  feature_cols=feature_cols_n2v_128,  target_col="log_systemic_risk_label")

print("GraphSAGE v1 32  —", trainer_sage_32.train_df.shape,  trainer_sage_32.val_df.shape)
print("GraphSAGE v1 64  —", trainer_sage_64.train_df.shape,  trainer_sage_64.val_df.shape)
print("GraphSAGE v1 128 —", trainer_sage_128.train_df.shape, trainer_sage_128.val_df.shape)
print("Node2Vec v1 32   —", trainer_n2v_32.train_df.shape,   trainer_n2v_32.val_df.shape)
print("Node2Vec v1 64   —", trainer_n2v_64.train_df.shape,   trainer_n2v_64.val_df.shape)
print("Node2Vec v1 128  —", trainer_n2v_128.train_df.shape,  trainer_n2v_128.val_df.shape)

GraphSAGE v1 32  — (109152, 37) (18192, 37)
GraphSAGE v1 64  — (109152, 69) (18192, 69)
GraphSAGE v1 128 — (109152, 133) (18192, 133)
Node2Vec v1 32   — (109152, 37) (18192, 37)
Node2Vec v1 64   — (109152, 69) (18192, 69)
Node2Vec v1 128  — (109152, 133) (18192, 133)


## Define Models

In [5]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train — GraphSAGE v1 (32-dim)

In [6]:
trainer_sage_32.train_all(candidate_models)
display(trainer_sage_32.leaderboard()[DISPLAY_COLS])
trainer_sage_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP,0.020,0.028,0.086,0.128
1,Random Forest,0.006,0.025,0.031,0.139
2,Gradient Boosting,0.010,0.025,0.055,0.156
3,XGBoost,0.010,0.026,0.051,0.158
4,Ridge,0.031,0.039,0.111,0.171
5,Linear Regression,0.031,0.039,0.111,0.171


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP,0.449,0.777,0.574,0.938
1,Random Forest,0.158,0.959,0.201,1.113
2,Gradient Boosting,0.259,1.150,0.332,1.305
3,XGBoost,0.224,1.184,0.291,1.328
4,Ridge,0.616,1.218,0.799,1.359
5,Linear Regression,0.615,1.218,0.799,1.359


## Train — GraphSAGE v1 (64-dim)

In [7]:
trainer_sage_64.train_all(candidate_models)
display(trainer_sage_64.leaderboard()[DISPLAY_COLS])
trainer_sage_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Ridge,0.034,0.038,0.133,0.161
1,Linear Regression,0.034,0.038,0.133,0.161
2,Random Forest,0.007,0.028,0.039,0.164
3,Gradient Boosting,0.012,0.029,0.065,0.189
4,XGBoost,0.010,0.030,0.056,0.195
5,MLP,0.026,0.037,0.127,0.207


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Ridge,0.819,1.059,1.024,1.204
1,Linear Regression,0.819,1.059,1.024,1.204
2,Random Forest,0.198,1.237,0.255,1.420
3,Gradient Boosting,0.343,1.501,0.445,1.678
4,XGBoost,0.288,1.535,0.373,1.751
5,MLP,0.650,1.507,0.828,1.823


## Train — GraphSAGE v1 (128-dim)

In [8]:
trainer_sage_128.train_all(candidate_models)
display(trainer_sage_128.leaderboard()[DISPLAY_COLS])
trainer_sage_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.007,0.030,0.040,0.177
1,XGBoost,0.010,0.031,0.056,0.195
2,Gradient Boosting,0.010,0.031,0.056,0.197
3,Ridge,0.033,0.041,0.142,0.201
4,Linear Regression,0.033,0.041,0.142,0.201
5,MLP,0.040,0.042,0.845,0.202


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.204,1.302,0.258,1.535
1,XGBoost,0.291,1.501,0.379,1.743
2,Gradient Boosting,0.275,1.544,0.380,1.774
3,Ridge,0.910,1.314,1.141,1.526
4,Linear Regression,0.910,1.314,1.141,1.526
5,MLP,0.890,1.372,1.123,1.584


## Train — Node2Vec v1 (32-dim)

In [9]:
trainer_n2v_32.train_all(candidate_models)
display(trainer_n2v_32.leaderboard()[DISPLAY_COLS])
trainer_n2v_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.007,0.030,0.032,0.129
1,Gradient Boosting,0.012,0.029,0.058,0.132
2,XGBoost,0.013,0.030,0.061,0.133
3,MLP,0.027,0.047,0.083,0.145
4,Linear Regression,0.053,0.077,0.133,0.192
5,Ridge,0.053,0.077,0.133,0.192


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.152,0.700,0.197,0.818
1,Gradient Boosting,0.202,0.763,0.298,0.859
2,XGBoost,0.216,0.782,0.318,0.871
3,MLP,0.420,0.876,0.530,0.976
4,Linear Regression,0.810,1.326,0.983,1.407
5,Ridge,0.810,1.326,0.983,1.407


## Train — Node2Vec v1 (64-dim)

In [10]:
trainer_n2v_64.train_all(candidate_models)
display(trainer_n2v_64.leaderboard()[DISPLAY_COLS])
trainer_n2v_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006,0.028,0.031,0.127
1,XGBoost,0.012,0.028,0.054,0.131
2,Gradient Boosting,0.016,0.029,0.072,0.133
3,MLP,0.025,0.042,0.069,0.135
4,Linear Regression,0.054,0.075,0.129,0.202
5,Ridge,0.054,0.075,0.129,0.202


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.150,0.730,0.196,0.818
1,XGBoost,0.175,0.772,0.247,0.866
2,Gradient Boosting,0.324,0.809,0.435,0.898
3,MLP,0.303,0.792,0.392,0.899
4,Linear Regression,0.776,1.372,0.932,1.445
5,Ridge,0.776,1.372,0.932,1.445


## Train — Node2Vec v1 (128-dim)

In [11]:
trainer_n2v_128.train_all(candidate_models)
display(trainer_n2v_128.leaderboard()[DISPLAY_COLS])
trainer_n2v_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006,0.028,0.032,0.122
1,XGBoost,0.011,0.029,0.049,0.131
2,Gradient Boosting,0.012,0.029,0.055,0.131
3,MLP,0.021,0.043,0.059,0.145
4,Ridge,0.053,0.083,0.129,0.194
5,Linear Regression,0.053,0.083,0.129,0.194


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.157,0.596,0.206,0.701
1,XGBoost,0.153,0.672,0.220,0.778
2,Gradient Boosting,0.213,0.687,0.298,0.795
3,MLP,0.243,0.891,0.342,1.015
4,Ridge,0.772,1.346,0.952,1.456
5,Linear Regression,0.772,1.346,0.952,1.456


## Top 1% Leaderboards

In [12]:
print("GraphSAGE v1 (32-dim)");  display(trainer_sage_32.leaderboard()[TOP1_COLS])
print("GraphSAGE v1 (64-dim)");  display(trainer_sage_64.leaderboard()[TOP1_COLS])
print("GraphSAGE v1 (128-dim)"); display(trainer_sage_128.leaderboard()[TOP1_COLS])
print("Node2Vec v1 (32-dim)");   display(trainer_n2v_32.leaderboard()[TOP1_COLS])
print("Node2Vec v1 (64-dim)");   display(trainer_n2v_64.leaderboard()[TOP1_COLS])
print("Node2Vec v1 (128-dim)");  display(trainer_n2v_128.leaderboard()[TOP1_COLS])

GraphSAGE v1 (32-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP,0.449,0.777,0.574,0.938
1,Random Forest,0.158,0.959,0.201,1.113
2,Gradient Boosting,0.259,1.150,0.332,1.305
3,XGBoost,0.224,1.184,0.291,1.328
4,Ridge,0.616,1.218,0.799,1.359
5,Linear Regression,0.615,1.218,0.799,1.359


GraphSAGE v1 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Ridge,0.819,1.059,1.024,1.204
1,Linear Regression,0.819,1.059,1.024,1.204
2,Random Forest,0.198,1.237,0.255,1.420
3,Gradient Boosting,0.343,1.501,0.445,1.678
4,XGBoost,0.288,1.535,0.373,1.751
5,MLP,0.650,1.507,0.828,1.823


GraphSAGE v1 (128-dim)


...

,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.152,0.700,0.197,0.818
1,Gradient Boosting,0.202,0.763,0.298,0.859
2,XGBoost,0.216,0.782,0.318,0.871
3,MLP,0.420,0.876,0.530,0.976
4,Linear Regression,0.810,1.326,0.983,1.407
5,Ridge,0.810,1.326,0.983,1.407


Node2Vec v1 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.150,0.730,0.196,0.818
1,XGBoost,0.175,0.772,0.247,0.866
2,Gradient Boosting,0.324,0.809,0.435,0.898
3,MLP,0.303,0.792,0.392,0.899
4,Linear Regression,0.776,1.372,0.932,1.445
5,Ridge,0.776,1.372,0.932,1.445


Node2Vec v1 (128-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.157,0.596,0.206,0.701
1,XGBoost,0.153,0.672,0.220,0.778
2,Gradient Boosting,0.213,0.687,0.298,0.795
3,MLP,0.243,0.891,0.342,1.015
4,Ridge,0.772,1.346,0.952,1.456
5,Linear Regression,0.772,1.346,0.952,1.456


## Hyperparameter Tuning

In [13]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_iter, cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_

RF_PARAMS = {
    "model__n_estimators":      [100, 200, 300],
    "model__max_depth":         [None, 5, 10],
    "model__min_samples_leaf":  [1, 2, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__max_features":      ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          [100, 200, 300],
    "model__max_depth":         [3, 5, 8, None],
    "model__learning_rate":     [0.005, 0.01, 0.05],
    "model__min_samples_leaf":  [5, 10, 20, 50, 100],
    "model__l2_regularization": [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__max_leaf_nodes":    [15, 20, 30, 40, 50, 60],
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     [100, 200, 400],
    "model__max_depth":        [3, 4, 5, 6, 8, 10],
    "model__learning_rate":    [0.005, 0.01, 0.05],
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda":       [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, "auto"],
}

In [14]:
for label, t in [
    ("GraphSAGE v1 (32-dim)",  trainer_sage_32),
    ("GraphSAGE v1 (64-dim)",  trainer_sage_64),
    ("GraphSAGE v1 (128-dim)", trainer_sage_128),
    ("Node2Vec v1 (32-dim)",   trainer_n2v_32),
    ("Node2Vec v1 (64-dim)",   trainer_n2v_64),
    ("Node2Vec v1 (128-dim)",  trainer_n2v_128),
]:
    print(f"\n===== Tuning {label} =====")
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=5, random_state=42)), MLP_PARAMS, "MLP (tuned)")
    display(t.leaderboard()[DISPLAY_COLS])
    display(t.leaderboard()[TOP1_COLS])


===== Tuning GraphSAGE v1 (32-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.020,0.026,0.096,0.122
1,MLP,0.020,0.028,0.086,0.128
2,Random Forest,0.006,0.025,0.031,0.139
3,Random Forest (tuned),0.008,0.024,0.044,0.139
4,Gradient Boosting (tuned),0.015,0.024,0.079,0.144
5,XGBoost (tuned),0.015,0.025,0.077,0.146
6,Gradient Boosting,0.010,0.025,0.055,0.156
7,XGBoost,0.010,0.026,0.051,0.158
8,Ridge,0.031,0.039,0.111,0.171
9,Linear Regression,0.031,0.039,0.111,0.171


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP (tuned),0.598,0.743,0.760,0.910
1,MLP,0.449,0.777,0.574,0.938
2,Random Forest,0.158,0.959,0.201,1.113
3,Random Forest (tuned),0.226,0.962,0.295,1.119
4,Gradient Boosting (tuned),0.421,1.025,0.530,1.171
5,XGBoost (tuned),0.425,1.057,0.535,1.190
6,Gradient Boosting,0.259,1.150,0.332,1.305
7,XGBoost,0.224,1.184,0.291,1.328
8,Ridge,0.616,1.218,0.799,1.359
9,Linear Regression,0.615,1.218,0.799,1.359



===== Tuning GraphSAGE v1 (64-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.027,0.029,0.127,0.134
1,Random Forest (tuned),0.019,0.025,0.108,0.154
2,Ridge,0.034,0.038,0.133,0.161
3,Linear Regression,0.034,0.038,0.133,0.161
4,Random Forest,0.007,0.028,0.039,0.164
5,Gradient Boosting (tuned),0.023,0.030,0.119,0.175
6,XGBoost (tuned),0.020,0.028,0.109,0.179
7,Gradient Boosting,0.012,0.029,0.065,0.189
8,XGBoost,0.010,0.030,0.056,0.195
9,MLP,0.026,0.037,0.127,0.207


...

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.008,0.028,0.040,0.127
1,Random Forest,0.006,0.028,0.031,0.127
2,MLP (tuned),0.017,0.033,0.066,0.128
3,XGBoost,0.012,0.028,0.054,0.131
4,XGBoost (tuned),0.011,0.028,0.050,0.132
5,Gradient Boosting,0.016,0.029,0.072,0.133
6,Gradient Boosting (tuned),0.015,0.028,0.073,0.133
7,MLP,0.025,0.042,0.069,0.135
8,Linear Regression,0.054,0.075,0.129,0.202
9,Ridge,0.054,0.075,0.129,0.202


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.173,0.736,0.234,0.818
1,Random Forest,0.150,0.730,0.196,0.818
2,MLP (tuned),0.272,0.714,0.360,0.816
3,XGBoost,0.175,0.772,0.247,0.866
4,XGBoost (tuned),0.158,0.813,0.213,0.898
5,Gradient Boosting,0.324,0.809,0.435,0.898
6,Gradient Boosting (tuned),0.331,0.809,0.443,0.896
7,MLP,0.303,0.792,0.392,0.899
8,Linear Regression,0.776,1.372,0.932,1.445
9,Ridge,0.776,1.372,0.932,1.445



===== Tuning Node2Vec v1 (128-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.010,0.028,0.051,0.121
1,Random Forest,0.006,0.028,0.032,0.122
2,Gradient Boosting (tuned),0.013,0.030,0.061,0.129
3,XGBoost (tuned),0.011,0.029,0.049,0.130
4,XGBoost,0.011,0.029,0.049,0.131
5,Gradient Boosting,0.012,0.029,0.055,0.131
6,MLP (tuned),0.021,0.036,0.070,0.132
7,MLP,0.021,0.043,0.059,0.145
8,Ridge,0.053,0.083,0.129,0.194
9,Linear Regression,0.053,0.083,0.129,0.194


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.237,0.606,0.323,0.709
1,Random Forest,0.157,0.596,0.206,0.701
2,Gradient Boosting (tuned),0.244,0.687,0.340,0.788
3,XGBoost (tuned),0.158,0.755,0.220,0.844
4,XGBoost,0.153,0.672,0.220,0.778
5,Gradient Boosting,0.213,0.687,0.298,0.795
6,MLP (tuned),0.291,0.683,0.387,0.806
7,MLP,0.243,0.891,0.342,1.015
8,Ridge,0.772,1.346,0.952,1.456
9,Linear Regression,0.772,1.346,0.952,1.456


## Optional — Inspect Best Hyperparameters / Save Models

In [15]:
# Display best hyperparameters found during tuning — run this cell when you want to inspect them
all_trainers = [
    ("GraphSAGE v1 (32-dim)",  trainer_sage_32),
    ("GraphSAGE v1 (64-dim)",  trainer_sage_64),
    ("GraphSAGE v1 (128-dim)", trainer_sage_128),
    ("Node2Vec v1 (32-dim)",   trainer_n2v_32),
    ("Node2Vec v1 (64-dim)",   trainer_n2v_64),
    ("Node2Vec v1 (128-dim)",  trainer_n2v_128),
]
for label, t in all_trainers:
    if t.best_params:
        print(f"\n{'='*50}\n{label}")
        for model_name, params in t.best_params.items():
            print(f"  {model_name}:")
            for k, v in params.items():
                print(f"    {k.replace('model__', '')}: {v}")


GraphSAGE v1 (32-dim)
  Random Forest (tuned):
    n_estimators: 200
    min_samples_split: 10
    min_samples_leaf: 1
    max_features: 1.0
    max_depth: None
  Gradient Boosting (tuned):
    min_samples_leaf: 50
    max_leaf_nodes: 15
    max_iter: 100
    max_depth: 8
    max_bins: 255
    learning_rate: 0.05
    l2_regularization: 1.0
  XGBoost (tuned):
    subsample: 0.9
    reg_lambda: 0.0001
    reg_alpha: 1.0
    n_estimators: 400
    min_child_weight: 10
    max_depth: 6
    learning_rate: 0.01
    gamma: 2.0
    colsample_bytree: 1.0
  MLP (tuned):
    learning_rate_init: 0.05
    learning_rate: constant
    hidden_layer_sizes: (256, 128, 64)
    batch_size: auto
    alpha: 0.001
    activation: relu

GraphSAGE v1 (64-dim)
  Random Forest (tuned):
    n_estimators: 300
    min_samples_split: 2
    min_samples_leaf: 15
    max_features: 0.8
    max_depth: 5
  Gradient Boosting (tuned):
    min_samples_leaf: 20
    max_leaf_nodes: 30
    max_iter: 200
    max_depth: 3
    max

In [16]:
# Save specific models to disk — edit the list below and run this cell when you want to persist them
from src.models.ml_train_and_store import load_model

SAVE_DIR = PROJECT_ROOT / "src" / "models" / "dataset_1" / "05_a"

for label, t in all_trainers:
    best_name = t.leaderboard().iloc[0]["model"]
    path = t.save_model(best_name, SAVE_DIR)

# To reload later:
# model = load_model(path)

Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\MLP_(tuned).joblib
Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\MLP_(tuned).joblib
Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\MLP_(tuned).joblib
Saved 'Random Forest (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\Random_Forest_(tuned).joblib
Saved 'Random Forest (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\Random_Forest_(tuned).joblib
Saved 'Random Forest (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\Random_Forest_(tuned).joblib


In [17]:
SAVE_DIR = PROJECT_ROOT / "src" / "models" / "dataset_1" / "05_a"

# Save only MLP (tuned) from GraphSAGE v1 32-dim
trainer_sage_32.save_model("MLP (tuned)", SAVE_DIR)


Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\train_saved\05_a\MLP_(tuned).joblib


WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/models/train_saved/05_a/MLP_(tuned).joblib')